In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kiet222005/assist2009/skill_builder_data_corrected_collapsed.csv


In [2]:
%%bash
set -e
set -o pipefail
cd /kaggle/working
[ -d pykt-toolkit ] || git clone -q https://github.com/pykt-team/pykt-toolkit.git
cd pykt-toolkit
pip install -q -e . && pip install -q einops matplotlib scipy tqdm

mkdir -p data/assist2009
python - <<'EOF'
src = "/kaggle/input/datasets/kiet222005/assist2009/skill_builder_data_corrected_collapsed.csv"
dst = "data/assist2009/skill_builder_data_corrected_collapsed.csv"
raw = open(src, "rb").read()
print("size:", len(raw))
print("first bytes:", raw[:200])
txt = None
for enc in ["utf-8", "cp1252", "latin-1"]:
    try:
        txt = raw.decode(enc); print("-> decode OK bằng:", enc); break
    except UnicodeDecodeError as e:
        print("-> không phải", enc, ":", e)
lines = txt.splitlines()
print("số dòng:", len(lines))
print("header:", lines[0][:300])
open(dst, "w", encoding="utf-8", newline="").write(txt)
EOF
ls -la data/assist2009

cd examples && mkdir -p logs
python -u data_preprocess.py --dataset_name=assist2009 2>&1 | tee logs/prep09.log | tail -12
echo "=== KIỂM TRA: phải là qs: 17737, cs: 123 ==="
grep -m2 "qs:" logs/prep09.log
ls ../data/assist2009

python -u wandb_dkt_train.py --dataset_name=assist2009 --fold=0 --seed=42 \
    --use_wandb=0 --add_uuid=0 --save_dir=sm_dkt09 > logs/dkt09_f0_s42_train.log 2>&1
tail -4 logs/dkt09_f0_s42_train.log

CKPT=$(dirname $(find sm_dkt09 -name "config.json" | head -n 1))
echo "CKPT=$CKPT"
python -u wandb_predict.py --save_dir=$CKPT --bz=256 --use_wandb=0 \
    --fusion_type=late_fusion > logs/dkt09_f0_s42_predict.log 2>&1 || true
grep -E "Final reported|windowauclate|windowacclate|window_testauc|Traceback|Error" logs/dkt09_f0_s42_predict.log | tail -15

size: 64412812
first bytes: b'"","order_id","assignment_id","user_id","assistment_id","problem_id","original","correct","attempt_count","ms_first_response","tutor_mode","answer_type","sequence_id","student_class_id","position","ty'
-> không phải utf-8 : 'utf-8' codec can't decode byte 0x80 in position 13450483: invalid start byte
-> decode OK bằng: cp1252
số dòng: 346861
header: "","order_id","assignment_id","user_id","assistment_id","problem_id","original","correct","attempt_count","ms_first_response","tutor_mode","answer_type","sequence_id","student_class_id","position","type","base_sequence_id","skill_id","skill_name","teacher_id","school_id","hint_count","hint_total","o
total 62912
drwxr-xr-x 2 root root     4096 Oct 22 11:43 .
drwxr-xr-x 4 root root     4096 Oct 22 11:43 ..
-rw-r--r-- 1 root root 64412822 Oct 22 11:43 skill_builder_data_corrected_collapsed.csv
dropnum: 2
original test interactions num: 53148, select num: 0, qs: 13454, cs: 122, seqnum: 770
test sequences interactio